# 🌐 LLM Gateway with Portkey: Zero to Production
### A Complete Hands-On Guide for Enterprise AI Systems

---

> **What you'll build:** A progressively enhanced LLM setup — starting from a raw direct API call, then routing every request through a production-grade gateway with observability, retries, fallbacks, caching, and more.

| Experiment | What We Build | New Concept |
|---|---|---|
| 🔴 Baseline | Raw Groq call, no gateway | The problem |
| 🟡 Exp 1 | Route through Portkey | Gateway basics |
| 🟡 Exp 2 | + Metadata & Observability | Request tagging |
| 🟡 Exp 3 | + Automatic Retries | Resilience |
| 🟡 Exp 4 | + Request Timeouts | Latency control |
| 🟢 Exp 5 | + Fallbacks | Multi-provider routing |
| 🟢 Exp 6 | + Load Balancing | Traffic splitting |
| 🟢 Exp 7 | + Caching | Cost & speed |
| 🟢 Exp 8 | + Saved Config from Dashboard | Production configs |
| 🏭 Exp 9 | LangChain Drop-in | RAG integration |
| 🏭 Exp 10 | Full Production Gateway | Everything combined |

---
# PART 1 — Theory: What Is an LLM Gateway and Why Do We Need It?

## The Problem With Direct LLM Calls

When you call an LLM API directly:

```python
# What most teams start with
response = groq_client.chat.completions.create(model="llama-3.3-70b-versatile", ...)
```

This works — until it doesn't. Real production problems:

| Problem | Impact |
|---|---|
| Groq rate-limits you at 3 AM | App breaks for hours with no fallback |
| You switch providers | Rewrite every API call across 10+ files |
| Nobody knows what prompts are sent | Zero visibility, impossible to debug |
| Same question asked 1000 times | You pay for 1000 LLM calls |
| Groq goes down | Full outage — no automatic recovery |
| Latency spikes | No timeout, users wait forever |

An LLM Gateway solves all of these with **zero changes to your business logic**.

## What Is a Gateway?

A gateway is a **proxy layer** between your application and any LLM API:

```
Your App  →  [LLM Gateway]  →  Groq / NVIDIA / OpenAI / Anthropic
```

The gateway intercepts every request and can:
1. **Route** to the right provider
2. **Retry** automatically on transient failures
3. **Fallback** to a different provider if the primary fails
4. **Balance** load across multiple models by weight
5. **Cache** responses so identical requests never hit the LLM twice
6. **Log** everything with full request/response detail
7. **Tag** requests with metadata (user, session, feature) for analytics
8. **Enforce** timeouts and kill slow requests

## Why Portkey?

| Feature | Portkey | LiteLLM | Direct SDK |
|---|---|---|---|
| 250+ models unified | ✅ | ✅ | ❌ one provider |
| Automatic fallbacks | ✅ | ✅ | ❌ manual |
| Request caching | ✅ | ✅ | ❌ manual |
| Observability dashboard | ✅ beautiful UI | ⚠️ basic | ❌ none |
| LangChain drop-in | ✅ | ✅ | ✅ native |
| Config-as-code | ✅ JSON/YAML | ❌ | ❌ |
| Open source | ✅ Apache 2.0 | ✅ MIT | N/A |
| Overhead | ~20–40ms | ~30ms | 0ms |

Portkey handles **25M+ daily requests** with **99.99% uptime**.

## How Portkey Sits in Your Stack

```
                    YOUR APPLICATION
                          |
                          |  standard API call
                          v
              +---------------------------+
              |     PORTKEY GATEWAY       |
              |                           |
              |  [1] Log request          |
              |  [2] Check cache          |
              |  [3] Apply config rules   |
              |      - retry policy       |
              |      - timeout            |
              |      - routing strategy   |
              |  [4] Route to provider    |
              +---------------------------+
               /           |          \
              v            v           v
           Groq        NVIDIA       OpenAI
        (primary)    (fallback)  (fallback 2)
              \____________|__________/
                           |
              +---------------------------+
              |  [5] Log response         |
              |  [6] Store in cache       |
              |  [7] Return to app        |
              +---------------------------+
                           |
                    YOUR APPLICATION
```

## Key Concept: Provider Slugs

When you add a provider integration in Portkey, you give it a **slug** — a short name.

```
Portkey Dashboard → Integrations → Add Groq → name it "flight-policy"
```

Then in your code, reference that provider as `@slug/model-name`:

```python
portkey.chat.completions.create(
    model="@flight-policy/llama-3.3-70b-versatile",
    ...
)
```

Portkey sees `@flight-policy`, looks up the stored Groq credentials, and forwards the request.
**Your API key never touches your code** — it lives in the Portkey dashboard.

In [1]:
import os
import time
import uuid
import json
from dotenv import load_dotenv
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

In [ ]:
load_dotenv(dotenv_path="../.env")

# Portkey API key — authenticates your app to the Portkey gateway
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY", "GPZu9vStlhnX25J9/F5YCUiJdwBT")

# Provider slug — the name you gave when adding Groq in the Portkey dashboard
# Model format: @<slug>/<model-name>
GROQ_SLUG    =  "rag"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "brag"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

# Keep GROQ_API_KEY for the LangChain experiment (Exp 9)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [4]:
# helper functions

def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")
    
    
def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

# Main Portkey client — used for all simple experiments
portkey = Portkey(api_key=PORTKEY_API_KEY)


print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")


Setup complete!
  Portkey API Key : OK
  Groq slug       : rag
  Groq model ref  : @rag/llama-3.3-70b-versatile
  Groq slug 2     : brag
  Small model ref : @brag/llama-3.1-8b-instant

Portkey Gateway : https://api.portkey.ai/v1





# BASELINE — Direct LLM Call (No Gateway)

**Goal:** See what a raw LLM call looks like — no routing, no logging, no resilience.

In [5]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)

section("BASELINE — Direct Groq Call")

questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time()-t0)*1000, label="direct Groq — no gateway")

print("\nMissing: logging, retries, fallback, caching, metadata.")


c:\Users\djadh\Downloads\okay\tenvv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱  265ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows a single physical device, such as a network interface card (NIC) or a storage controller, to appear as multiple virtual devices to the operating system and applications. This is achieved...
⏱  1552ms | direct Groq — no gateway
──────────────────────────────────────────────────────────────

Missing: logging, retries, fallback, caching, metadata.


---
# EXPERIMENT 1 — Route Through the Gateway

**Goal:** Same call, same answer — but now every request is logged in your Portkey dashboard.

**New concepts:**
- `Portkey(api_key=...)` — the gateway client
- `model="@slug/model-name"` — tells Portkey which provider to use
- `response.choices[0].message.content` — standard OpenAI response format

In [6]:
section("EXP 1 — Basic Gateway Call")


questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]

for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")

print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


  EXP 1 — Basic Gateway Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱  2354ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows a single physical device, such as a network interface card (NIC) or a storage controller, to appear as multiple virtual devices to the operating system and applications. This is achieved...
⏱  2170ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

✅ Check portkey.ai → Logs to see both requests fully logged!
   Token count, cost, latency — all tracked. Zero extra code.


### What changed?

```
Before:  Your App  →  Groq directly
After:   Your App  →  Portkey  →  Groq (via @flight-policsy integration)
```

- Portkey adds ~20–40ms then forwards to Groq using stored credentials
- Full request + response logged automatically — no extra code
- **You changed 3 lines. Your business logic is identical.**

---
# EXPERIMENT 2 — Metadata & Observability

**Goal:** Tag every request with user, session, and feature info so you can filter and analyse in the dashboard.

**New concept:** `portkey.with_options(metadata={...})` — attaches tags to a single request.
The special key `_user` powers per-user analytics.

In [7]:
section("EXP 2 — Metadata & Observability")

session = str(uuid.uuid4())[:8]


scenarios = [
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "docs-chatbot",     "How does BGP path selection work?"),
    ("carol", "support-bot",      "What is SRIOV virtualization?"),
    ("alice", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   # same user, diff Q
]

for user, feature, q in scenarios:
    t0 = time.time()
    r = portkey.with_options(
        metadata={
            "_user":       user,          # powers per-user analytics in dashboard
            "session_id":  session,
            "feature":     feature,
            "environment": "notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")

print("\n✅ Dashboard now shows: cost per user, calls per feature, session grouping")


  EXP 2 — Metadata & Observability

👤 alice    | 🔧 enterprise-rag     | 3402ms
  Q: What is Kubernetes RBAC?
  A: Kubernetes Role-Based Access Control (RBAC) is a security feature that allows administrators to control access to cluste...

👤 bob      | 🔧 docs-chatbot       | 3017ms
  Q: How does BGP path selection work?
  A: BGP (Border Gateway Protocol) path selection is the process by which a BGP router chooses the best path to forward packe...

👤 carol    | 🔧 support-bot        | 2081ms
  Q: What is SRIOV virtualization?
  A: SR-IOV (Single Root I/O Virtualization) is a specification for a type of I/O virtualization technology developed by the ...

👤 alice    | 🔧 enterprise-rag     | 1619ms
  Q: Explain Kubernetes NetworkPolicy
  A: **Kubernetes NetworkPolicy**

Kubernetes NetworkPolicy is a feature that allows you to contro...

✅ Dashboard now shows: cost per user, calls per feature, session grouping


### Why metadata matters in production

With `_user` tagging you can answer:
- *Which users generate the most cost?*
- *Which feature uses the most tokens?*
- *What did Alice's full session look like?*
- *Is the RAG pipeline slower than the support bot?*

All answered from the Portkey dashboard — no extra logging code.

---
# EXPERIMENT 3 — Automatic Retries

**Goal:** Portkey automatically retries failed requests with exponential backoff. App code never sees transient 429/5xx errors.

**New concept:** Pass a `config` dict to `Portkey(...)` with retry settings.

In [ ]:
retry_config = {
    "retry": {
        "attempts": 3,
        "on_status_codes": [429, 500, 502, 503, 504]
    }
}

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, config=retry_config)


section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a Kubernetes DaemonSet?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")


  EXP 3 — Automatic Retries
Config: 3 retry attempts on [429, 500, 502, 503, 504]
Retries fire automatically on failure — transparent to your code

✅ Succeeded in 1945ms
   A Kubernetes DaemonSet is a type of workload that runs a specific container on each node in a cluster, or on a specific subset of nodes. It ensures that a certain pod (or container) is always running on each node, and automatically recreates the pod if it is terminated or deleted.

Here are some key

Retry sequence if Groq had failed:
  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3
  Your code only sees the final success or the last failure


---
# EXPERIMENT 4 — Request Timeouts

**Goal:** Kill requests that take too long. An LLM stall blocks your FastAPI worker indefinitely without a timeout.

**New concept:** `request_timeout` in milliseconds in the config dict. Portkey returns **HTTP 408** on timeout. Pair with fallbacks for auto-recovery.

In [9]:
# Config-level timeout (cleanest approach — works inside any routing config)
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms


portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config=timeout_config)

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")


print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))


  EXP 4 — Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.

✅ Response in 1032ms (within 10s timeout)
   Kubernetes networking provides a robust and flexible way to manage communication between pods, services, and external networks, allowing pods to communicate with each other and with services through a virtual network overlay. The Kubernetes networking model consists of several components, including pod networks, services, ingress controllers, and network policies, which work together to enable secure and efficient communication between applications running in a Kubernetes cluster.

--- Combining timeout + retry (production pattern) ---
{
  "request_timeout": 10000,
  "retry": {
    "attempts": 2,
    "on_status_codes": [
      408,
      429,
      503
    ]
  }
}


---
# EXPERIMENT 5 — Fallbacks

**Goal:** If the primary Groq model fails, automatically switch to the smaller Groq fallback. Users never see the failure.

**New concept:** `strategy.mode = "fallback"` with an ordered `targets` list. First target is primary, rest are fallbacks.
Both targets use Groq — one large model (70b), one small model (8b).

In [11]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},     # primary
        {"override_params": {"model": GROQ_MODEL_SMALL}}    # fallback if primary fails
    ]
}

portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)


section("EXP 5 — Fallback Routing")
print(f"Strategy: {GROQ_MODEL} → {GROQ_MODEL_SMALL} on failure\n")


fallback_questions = [
    "What is Intel QuickAssist Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="fallback ready")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")


print("\nFallback chain:")
print("  Groq 2xx   → return immediately")
print("  Groq 4xx/5xx → try small Groq model (8b) automatically")
print("  Check Portkey Logs to see fallback activations")



  EXP 5 — Fallback Routing
Strategy: @rag/llama-3.3-70b-versatile → @brag/llama-3.1-8b-instant on failure


──────────────────────────────────────────────────────────────
Q: What is Intel QuickAssist Technology?
A: Intel QuickAssist Technology (IQAT) is a set of instructions and hardware accelerators developed by Intel to accelerate specific tasks, particularly those related to cryptography, compression, and encryption. The technology is designed to offload computational...
⏱  2182ms | fallback ready
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: Explain Kubernetes persistent volume claims.
A: **Kubernetes Persistent Volume Claims (PVCs)**

Kubernetes Persistent Volume Claims (PVCs) are a way to request storage resources from a cluster. They provide a flexible and scalable way to manage persistent data i...
⏱  2155ms | fallback ready
──────────────────────────────────────────────────────────────

Fallba

### Narrow the fallback trigger

Default fires on any non-2xx. Narrow it to avoid accidental fallbacks on bad requests:

```python
# Only fall back on rate limits and server errors
"strategy": {"mode": "fallback", "on_status_codes": [429, 503]}
```

---
# EXPERIMENT 6 — Load Balancing

**Goal:** Split traffic between providers by weight. Use for gradual migration, A/B testing, or cost control.

**New concept:** `strategy.mode = "loadbalance"` with `weight` per target. Portkey normalises weights to percentages.

In [13]:
load_balance_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL},   "weight": 0.7},   # 70%
        {"override_params": {"model": GROQ_MODEL_SMALL}, "weight": 0.3}    # 30%
    ]
}

portkey_lb = Portkey(api_key=PORTKEY_API_KEY, config=load_balance_config)

section("EXP 6 — Load Balancing (70% large / 30% small)")


lb_questions = [
    "What is a Kubernetes Ingress resource?",
    "How does OSPF differ from BGP?",
    "What is Intel FPGA acceleration?",
    "Explain Kubernetes HPA.",
    "What is a VLAN trunk?",
    "How does Kubernetes etcd work?",
]

print("Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).\n")


for i, q in enumerate(lb_questions, 1):
    try:
        t0 = time.time()
        r = portkey_lb.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        print(f"Req {i} [{ms:.0f}ms]: {q}")
        print(f"         {r.choices[0].message.content[:120]}...")
    except Exception as e:
        print(f"Req {i}: ERROR — {e}")

print("\n✅ Check Portkey Logs to see which provider served each request")
print("   Set weight=0 to pause a target without removing it from the config")



  EXP 6 — Load Balancing (70% large / 30% small)
Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).

Req 1 [2063ms]: What is a Kubernetes Ingress resource?
         A Kubernetes Ingress resource is a Kubernetes object that provides a single entry point for incoming HTTP requests to ac...
Req 2 [2770ms]: How does OSPF differ from BGP?
         OSPF (Open Shortest Path First) and BGP (Border Gateway Protocol) are two different routing protocols used in computer n...
Req 3 [1931ms]: What is Intel FPGA acceleration?
         Intel FPGA (Field-Programmable Gate Array) acceleration refers to the use of programmable integrated circuits, known as ...
Req 4 [2478ms]: Explain Kubernetes HPA.
         **Kubernetes Horizontal Pod Autoscaling (HPA)**

Kubernetes Horizontal Pod...
Req 5 [1870ms]: What is a VLAN trunk?
         A VLAN (Virtual Local Area Network) trunk is a connection between two network devices, such as switches or routers, that...
Req 6 [19

### Load balancing use cases

| Scenario | Config |
|---|---|
| **Gradual migration** | Start 95/5, shift to 50/50, then 0/100 over weeks |
| **A/B testing** | 50/50 — measure quality + user satisfaction per model |
| **Cost control** | Route more traffic to the cheaper model |
| **Maintenance** | `weight: 0` pauses a target without removing it |

---
# EXPERIMENT 7 — Request Caching

**Goal:** Cache responses to identical questions. Second call is instant and costs nothing.

**New concept:** `cache.mode = "simple"` does exact-match caching. Identical request → served from cache, no LLM call.

In [ ]:
cache_config = {"cache": {"mode": "semantic"}}


portkey_cached = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)


section("EXP 7 — Semantic Caching")


# Use a short, precise question with temperature=0 to maximise cache-key stability
q = "Define Kubernetes autoscaling."
call_params = dict(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
    max_tokens=120,
)

print("--- CALL 1: Cache MISS — Portkey forwards to Groq ---")
t0 = time.time()
r1 = portkey_cached.chat.completions.create(**call_params)
t1 = (time.time() - t0) * 1000
ans1 = r1.choices[0].message.content.strip()
print(f"Answer  : {ans1[:200]}")
print(f"Latency : {t1:.0f}ms | Cost: normal token price")






  EXP 7 — Simple Caching
--- CALL 1: Cache MISS — Portkey forwards to Groq ---
Answer  : Kubernetes autoscaling is a feature in Kubernetes that automatically adjusts the number of replicas (i.e., copies) of a pod or a deployment based on the current workload or resource utilization. The g
Latency : 2634ms | Cost: normal token price


In [17]:
q = "what is Kubernetes autoscaling."
call_params = dict(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": q}],
    temperature=0,
    max_tokens=120,
)


In [18]:
print("CALL 2: Same request — should be a Cache HIT")
t0 = time.time()
r2 = portkey_cached.chat.completions.create(**call_params)
t2 = (time.time() - t0) * 1000
ans2 = r2.choices[0].message.content.strip()
print(f"Answer  : {ans2[:200]}")
print(f"Latency : {t2:.0f}ms")

CALL 2: Same request — should be a Cache HIT
Answer  : Kubernetes autoscaling is a feature that allows you to automatically adjust the number of replicas (i.e., copies) of a pod or a deployment in a Kubernetes cluster based on certain conditions, such as 
Latency : 1048ms
